<a href="https://colab.research.google.com/github/damayanthij42-44/QuantumShield-AI/blob/main/research_paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1:
Install REquired Libraries

In [ ]:
!pip install -q langchain langchain-community langchain-groq
!pip install -q pypdf chromadb sentence-transformers
!pip install -q langchain-huggingface

Step 2:
Upload Research Paper


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Research paper uploaded successfully!")
print("Number of papers uploaded:", len(uploaded))

Step 3:
Load Research Papers


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
all_documents = []
for filename in uploaded.keys():
    loader = PyPDFLoader(filename)
    documents = loader.load()
    all_documents.extend(documents)
print("Research papers loaded successfully!")
print("Total pages loaded:", len(all_documents))

In [ ]:
!pip install -q langchain-text-splitters

# Step 4:Text Chunking
      I split the research papers into smaller overlapping chunks so that relevant information can be retrieved more accuratelt.




In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(all_documents)
print("Text chunking completed successfully!")
print("Total chunks created:", len(chunks))

# Step 5: Create Embeddings
      I convert the research paper chunks into numerical vectors using a sentence embedding model



In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-l6-v2"
)
print("Embedding model loaded successfully")

Step 5 continued: Generate Embeddings

In [ ]:
embedded_chunks = embedding_model.embed_documents(
    [chunk.page_content for chunk in chunks]
)
print("Embeddings created sucessfully!")
print("Number of embedding:", len(embedded_chunks))
print("Embedding dimension:", len(embedded_chunks[0]))

# Step 6:  Create ChromaDB Vector Database
    I store the research paper chunk and their embeddings in ChromaDB. This allows the chatbot to quickly retrive the most relevant content when a user asks a question

   



In [ ]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name = "research_papers"
)
print("ChromaDB vecor database created successfully!")
print("Total chunks stored:", len(chunks))

Step 7: Retrieve Relevant Documents

    I Retrieve the most relevant research paper chunks from ChromaDB for a User's question.

In [ ]:
query = "What is the Transformer architecture?"
retrieved_docs = vectorstore.similarity_search(
    query,
    k=3
)
print("Relevant docunments retrieved successfully!")
print("Number of docunments retrieved:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs, 1):
  print(f"\n--- Result {i} ---")
  print("Source:", doc.metadata.get("source", "Unknown"))
  print("Page:", doc.metadata.get("Page", "Unknown"))
  print("Content:")
  print(doc.page_content[:500])

# Step 8 : Content Groq LLM

    I use the Groq LLM to generate answer from the relevant research paper content retrieved from ChromaDB.

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0
)
print("Groq LLM connected successfully!")

# Step 9:  Build the RAG Prompt
    I combine the retrieved research paper chunk with the user's question and send them to the Groq LLM.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a research paper question-answering assistant.

Answer the user's question ONLY using the provided research paper context.

If the answer is not available in the context, respond exactly:
"The answer is not available in the provided research papers."

Do not use outside knowledge.
Keep the answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
""")

print("RAG prompt created successfully!")

#Step 10:  Generate the RAG Answer
    I combine the retrieved research paper chunks with the user's question and send them to the Groq LLM.

In [ ]:
context = "\n\n".join(
    [
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Page: {doc.metadata.get('page', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    ]
)
question = query
messages = prompt.format_messages(
    context=context,
    question=question
)
response = llm.invoke(messages)
print("RAG answer generatd successfully!")
print("\nQuestion:", question)
print("\nAnswet:", response.content)

# Step 11: Create the Question Answering Function
    I create a reusable function that retrievers relevent reserach paper chunks and generates an answer using the Groq LLM.

In [ ]:
def ask_question(question):
  retrieved_docs = vectorstore.similarity_search(
      question,
      k=5
  )
  context = "\n\n".join(
      [
          f"Source: {doc.metadata.get('source', 'Unknown')}\n"
          f"page: {doc.metadata.get('page', 'Unknown')}\n"
          f"content: {doc.page_content}"
          for doc in retrieved_docs
      ]
  )
  messages = prompt.format_messages(
      context = context,
      question= question
  )
  response = llm.invoke(messages)
  return response.content

In [ ]:
answer = ask_question("What is sentence-BERT?")
print("Answer:")
print(answer)

# Step 12: Add Relevance Check
    I check whether the retrieved research paper content is relevant to the user's question.


In [ ]:
import numpy as np

print("NumPy loaded successfully!")

In [ ]:
def ask_question(question):
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=5
    )
    question_embedding = embedding_model.embed_query(question)
    relevant_docs = []
    for doc in retrieved_docs:
        doc_embedding = embedding_model.embed_query(
            doc.page_content
        )
        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding) *
            np.linalg.norm(doc_embedding)
        )
        if similarity >= 0.30:
            relevant_docs.append(doc)
    if not relevant_docs:
        return "The answer is not available in the provided research papers."
    context = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n"
            f"Page: {doc.metadata.get('page', 'Unknown')}\n"
            f"Content: {doc.page_content}"
            for doc in relevant_docs
        ]
    )
    messages = prompt.format_messages(
        context=context,
        question=question
    )
    response = llm.invoke(messages)

    return response.content

In [ ]:
print(ask_question("What is the recipe for making biryani?"))

In [ ]:
print(ask_question("What is the Transformer architecture?"))

# Step 13: I Display Supporting Sources
    I Display the research paper source and page number of the retrieved chunks used to answer the user's question.

In [ ]:
def get_supporting_sources(question):

    retrieved_docs = vectorstore.similarity_search(
        question,
        k=5
    )

    question_embedding = embedding_model.embed_query(question)

    relevant_docs = []

    for doc in retrieved_docs:

        doc_embedding = embedding_model.embed_query(
            doc.page_content
        )

        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding) *
            np.linalg.norm(doc_embedding)
        )

        if similarity >= 0.30:
            relevant_docs.append((doc, similarity))

    if not relevant_docs:
        return []

    print("Supporting Sources:\n")

    for i, (doc, score) in enumerate(relevant_docs[:3], 1):
        print(f"Source {i}")
        print("Paper:", doc.metadata.get("source", "Unknown"))
        print("Page:", doc.metadata.get("page", "Unknown"))
        print("Relevance Score:", round(score, 4))
        print("-" * 50)

    return relevant_docs

In [ ]:
soueces = get_supporting_sources(
    "What is the transformer architecture"
)

# Step 14: Generate Anwer with Supporting Sources
    I combinr the RAG answer with the relevant research paper sources

In [ ]:
def answer_with_sources(question):
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=5
    )
    question_embedding = embedding_model.embed_query(question)
    relevant_docs = []
    for doc in retrieved_docs:
        doc_embedding = embedding_model.embed_query(
            doc.page_content
        )
        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding) *
            np.linalg.norm(doc_embedding)
        )
        if similarity >= 0.30:
            relevant_docs.append((doc, similarity))
    if not relevant_docs:
        print("Answer:")
        print("The answer is not available in the provided research papers.")
        return
    context = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n"
            f"Page: {doc.metadata.get('page', 'Unknown')}\n"
            f"Content: {doc.page_content}"
            for doc, score in relevant_docs
        ]
    )
    messages = prompt.format_messages(
        context=context,
        question=question
    )
    response = llm.invoke(messages)
    print("Answer:")
    print(response.content)
    print("\nSupporting Sources:")
    for i, (doc, score) in enumerate(relevant_docs, 1):
        print(f"\nSource {i}")
        print("Paper:", doc.metadata.get("source", "Unknown"))
        print("Page:", doc.metadata.get("page", "Unknown"))
        print("Relevance Score:", round(score, 4))

In [ ]:
answer_with_sources(
    "What is REALM"
)

In [ ]:
answer_with_sources(
    "what is the recipe for making sambar making"
)

# Step 15: Evaluate the RAG system
    I test the chatbot with 10 question covering all five research papers.

In [ ]:
evaluation_questions = [
    "What is the Transformer architecture?",
    "What is multi-head attention?",

    "What is BERT?",
    "How is BERT pre-trained?",

    "What is Retrieval-Augmented Generation?",
    "Why does RAG use a retriever?",

    "What is Sentence-BERT?",
    "What is the purpose of Siamese networks in Sentence-BERT?",

    "What is REALM?",
    "What is the recipe for making biryani?"
]

print("Total evaluation questions:", len(evaluation_questions))

# Step 16: Improved Question Answering Function
    I improve the question-answering function by first identifying the relevant research paper and then retrieving chunks only from that paper.

In [ ]:
def improved_ask_question(question):
    paper_keywords = {
        "Sentence-BERT": [
            "sentence-bert",
            "sentence bert",
            "sbert",
            "siamese bert"
        ],
        "RAG": [
            "retrieval-augmented generation",
            "rag"
        ],
        "REALM": [
            "realm",
            "retrieval-augmented language model"
        ],
        "BERT": [
            "bert"
        ],
        "Transformer": [
            "transformer",
            "attention is all you need",
            "multi-head attention"
        ]
    }
    question_lower = question.lower()
    selected_paper = None
    for paper, keywords in paper_keywords.items():
        if any(keyword in question_lower for keyword in keywords):
            selected_paper = paper
            break
    if selected_paper is None:
        return "The answer is not available in the provided research papers."
    paper_ids = {
        "Sentence-BERT": "1908.10084",
        "RAG": "2005.11401",
        "REALM": "2002.08909",
        "BERT": "1810.04805",
        "Transformer": "1706.03762"
    }
    paper_id = paper_ids[selected_paper]
    selected_docs = [
        doc for doc in chunks
        if paper_id in doc.metadata.get("source", "")
    ]
    if not selected_docs:
        return "The answer is not available in the provided research papers."
    question_embedding = embedding_model.embed_query(question)
    scored_docs = []
    for doc in selected_docs:
        doc_embedding = embedding_model.embed_query(doc.page_content)
        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding) *
            np.linalg.norm(doc_embedding)
        )
        scored_docs.append((doc, similarity))
    scored_docs.sort(
        key=lambda x: x[1],
        reverse=True
    )
    top_docs = [
        doc for doc, score in scored_docs[:5]
        if score >= 0.30
    ]
    if not top_docs:
        return "The answer is not available in the provided research papers."
    context = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n"
            f"Page: {doc.metadata.get('page', 'Unknown')}\n"
            f"Content: {doc.page_content}"
            for doc in top_docs
        ]
    )
    messages = prompt.format_messages(
        context=context,
        question=question
    )
    response = llm.invoke(messages)
    answer = response.content.strip()
    if not answer:
        response = llm.invoke(messages)
        answer = response.content.strip()
    if not answer:
        return "The answer is not available in the provided research papers."
    return answer

In [ ]:
print(improved_ask_question("How is BERT pre-trained?"))

In [ ]:
question = "What is Sentence-BERT?"
answer = improved_ask_question(question)
print("Question:", question)
print("\nAnswer:")
print(answer)

In [ ]:
question = "What is BERT?"
answer = improved_ask_question(question)
print("Question:", question)
print("\nAnswer:")
print(answer)

In [ ]:
test_questions = [
    "What is Retrieval-Augmented Generation?",
    "What is REALM?"
]
for question in test_questions:
    print("=" * 70)
    print("Question:", question)
    print("=" * 70)
    answer = improved_ask_question(question)
    print("Answer:")
    print(answer)
    print()

In [ ]:
test_questions = [
    "What is the Transformer architecture?",
    "What is the recipe for making biryani?"
]
for question in test_questions:
    print("=" * 70)
    print("Question:", question)
    print("=" * 70)
    answer = improved_ask_question(question)
    print("Answer:")
    print(answer)
    print()

# Step 18: Final Evaluation of the Improved RAG System
    I evaluate the improved RAG system using 10 questions covering all five research papers.

In [ ]:
evaluation_questions = [
    "What is the Transformer architecture?",
    "What is multi-head attention?",

    "What is BERT?",
    "How is BERT pre-trained?",

    "What is Retrieval-Augmented Generation?",
    "Why does RAG use a retriever?",

    "What is Sentence-BERT?",
    "What is the purpose of Siamese networks in Sentence-BERT?",

    "What is REALM?",
    "What is the recipe for making biryani?"
]

print("Total evaluation questions:", len(evaluation_questions))

In [ ]:
for i, question in enumerate(evaluation_questions, 1):
    print("=" * 70)
    print(f"Question {i}: {question}")
    print("=" * 70)
    answer = improved_ask_question(question)
    print("Answer:")
    print(answer)
    print()

In [ ]:
for i, question in enumerate(evaluation_questions, 1):
    print("=" * 70)
    print(f"Question {i}: {question}")
    print("=" * 70)
    answer = improved_ask_question(question)
    print("Answer:")
    print(answer)
    print()

# Step 19: Display Top-3 Supporting Sources

I display the answer along with the top 3 relevant research paper passages.

In [ ]:
def get_supporting_sources(question):
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=5
    )
    question_embedding = embedding_model.embed_query(question)
    relevant_docs = []
    for doc in retrieved_docs:
        doc_embedding = embedding_model.embed_query(
            doc.page_content
        )
        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding) *
            np.linalg.norm(doc_embedding)
        )
        if similarity >= 0.30:
            relevant_docs.append((doc, similarity))
    if not relevant_docs:
        return []
    print("Top 3 Supporting Sources:\n")
    for i, (doc, score) in enumerate(relevant_docs[:3], 1):
        print(f"Source {i}")
        print("Paper:", doc.metadata.get("source", "Unknown"))
        print("Page:", doc.metadata.get("page", "Unknown"))
        print("Relevance Score:", round(score, 4))
        print("-" * 50)
    return relevant_docs

In [ ]:
sources = get_supporting_sources(
    "What is the Transformer architecture?"
)

# Step 20: Build Streamlit User Interface
    I build a simple Streamlit interface for the research paper chatbot.

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
from getpass import getpass
from pyngrok import ngrok
ngrok_token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(ngrok_token)
print("ngrok configured successfully!")

In [ ]:
%%writefile app.py
import streamlit as st
st.set_page_config(
    page_title="Research Paper Answer Bot",
    page_icon=" "
)
st.title("Research Paper Answer Bot")
st.write(
    "Ask questions based on the provided research papers."
)
question = st.text_input(
    "Enter your question:"
)
if question:
    st.write("Your question:", question)

# Step 21: Run the Strealit Application
    I run the Streamlit application on a local port in Google Colab

In [ ]:
!streamlit run app.py &> /content/streamlit.log &

# Step 21: Check Streamlit Application
    I check the Streamlit log to verify that the application started successfully.

In [ ]:
!cat /content/streamlit.log

In [ ]:
!pip install -q pyngrok

# Step 22: Create Public URL Using ngrok
    I use ngrok to create a public URL for the Streamlit application running on port 8502.

In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("Streamlit App URL:")
print(public_url)

In [ ]:
print(public_url)

# Step 23: Connect RAG Pipeline with Streamlit
    I connect the final RAG pipeline with the Streamlit interface.

In [ ]:
%%writefile app.py

import os
import glob
import numpy as np
import streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
st.set_page_config(
    page_title="Research Paper Answer Bot",
    page_icon="📚",
    layout="wide"
)

st.markdown("""
<style>

.stApp {
    background-color: #f7f4ff;
    color: #35206b;
}

h1, h2, h3, h4, p, label {
    color: #35206b !important;
}

.main-title {
    font-size: 38px;
    font-weight: 700;
    color: #35206b !important;
    margin-bottom: 5px;
}

.subtitle {
    font-size: 17px;
    color: #6f6590 !important;
    margin-bottom: 25px;
}

.answer-box {
    background-color: #ffffff;
    color: #35206b !important;
    padding: 22px;
    border-radius: 16px;
    border: 1px solid #e6ddff;
    box-shadow: 0 4px 15px rgba(90, 60, 150, 0.08);
}

.answer-box * {
    color: #35206b !important;
}

.source-box {
    background-color: #eee7ff;
    color: #35206b !important;
    padding: 15px;
    border-radius: 12px;
    margin-top: 10px;
}

.source-box * {
    color: #35206b !important;
}

div[data-baseweb="input"] {
    background-color: #ffffff !important;
}

div[data-baseweb="input"] input {
    color: #35206b !important;
    background-color: #ffffff !important;
}

div[data-baseweb="input"] input::placeholder {
    color: #8b80aa !important;
}

/* SOURCE EXPANDER */

[data-testid="stExpander"] summary {
    background-color: #eee7ff !important;
    color: #35206b !important;
    border: 1px solid #e6ddff !important;
    border-radius: 10px !important;
}

[data-testid="stExpander"] summary p {
    color: #35206b !important;
}

[data-testid="stExpander"] div[data-testid="stExpanderDetails"] {
    background-color: #f7f4ff !important;
    color: #35206b !important;
}

[data-testid="stExpander"] div[data-testid="stExpanderDetails"] * {
    color: #35206b !important;
}

</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_rag():

    pdf_files = glob.glob("/content/*.pdf")

    all_documents = []

    for filename in pdf_files:

        loader = PyPDFLoader(filename)

        documents = loader.load()

        all_documents.extend(documents)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = text_splitter.split_documents(
        all_documents
    )

    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name="streamlit_research_papers"
    )
    llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0
    )
    return (
        chunks,
        embedding_model,
        vectorstore,
        llm
    )

chunks, embedding_model, vectorstore, llm = load_rag()

prompt = ChatPromptTemplate.from_template("""
You are a research paper question-answering assistant.

Answer the user's question ONLY using the provided
research paper context.

If the answer is not available in the context,
respond exactly:

"The answer is not available in the provided research papers."

Do not use outside knowledge.

Keep the answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
""")

def ask_question(question):

    paper_keywords = {

        "Sentence-BERT": [
            "sentence-bert",
            "sentence bert",
            "sbert",
            "siamese bert"
        ],

        "RAG": [
            "retrieval-augmented generation",
            "retrieval augmented generation",
            "rag"
        ],

        "REALM": [
            "realm",
            "retrieval-augmented language model",
            "retrieval augmented language model"
        ],

        "BERT": [
            "bert",
            "bidirectional encoder representations"
        ],

        "Transformer": [
            "transformer",
            "attention is all you need",
            "multi-head attention",
            "multi head attention"
        ]
    }


    question_lower = question.lower()

    selected_paper = None
    for paper, keywords in paper_keywords.items():

        if any(
            keyword in question_lower
            for keyword in keywords
        ):

            selected_paper = paper

            break

    if selected_paper is None:

        return (
            "The answer is not available in the "
            "provided research papers."
        ), []

    paper_ids = {

        "Sentence-BERT": "1908.10084",
        "RAG": "2005.11401",
        "REALM": "2002.08909",
        "BERT": "1810.04805",
        "Transformer": "1706.03762"
    }

    paper_names = {

        "1706.03762":
            "Attention Is All You Need",
        "1810.04805":
            "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "2005.11401":
            "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "1908.10084":
            "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks",
        "2002.08909":
            "REALM: Retrieval-Augmented Language Model Pre-Training"
    }

    paper_id = paper_ids[selected_paper]
    selected_docs = [

        doc

        for doc in chunks

        if paper_id in doc.metadata.get(
            "source",
            ""
        )
    ]

    if not selected_docs:

        return (
            "The answer is not available in the "
            "provided research papers."
        ), []

    question_embedding = (
        embedding_model.embed_query(question)
    )

    scored_docs = []
    for doc in selected_docs:
        doc_embedding = (
            embedding_model.embed_query(
                doc.page_content
            )
        )
        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding)
            *
            np.linalg.norm(doc_embedding)
        )


        scored_docs.append(
            (doc, similarity)
        )

    scored_docs.sort(
        key=lambda x: x[1],
        reverse=True
    )

    relevant_docs = [
        (doc, score)
        for doc, score in scored_docs[:5]
        if score >= 0.30
    ]

    if not relevant_docs:
        return (
            "The answer is not available in the "
            "provided research papers."
        ), []

    context = "\n\n".join(

        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n"
            f"Page: {doc.metadata.get('page', 'Unknown')}\n"
            f"Content: {doc.page_content}"

            for doc, score in relevant_docs
        ]

    )

    messages = prompt.format_messages(
        context=context,
        question=question

    )

    response = llm.invoke(messages)
    answer = response.content.strip()
    if not answer:

        return (
            "The answer is not available in the "
            "provided research papers."
        ), []
    sources = []
    for doc, score in relevant_docs[:3]:

        source_file = doc.metadata.get(
            "source",
            "Unknown"
        )

        paper_title = next(

            (
                title
                for paper_id, title
                in paper_names.items()
                if paper_id in source_file

            ),

            "Unknown Research Paper"
        )
        sources.append(
            (
                doc,
                score,
                paper_title
            )
        )

    return answer, sources

st.markdown(

    '<div class="main-title">'
    '📚 Research Paper Answer Bot'
    '</div>',

    unsafe_allow_html=True
)


st.markdown(

    '<div class="subtitle">'
    'Ask questions from your research papers and get '
    'accurate, source-based answers.'
    '</div>',

    unsafe_allow_html=True
)
question = st.text_input(

    "Ask your question",

    placeholder="Example: What is BERT?"

)

if question:
    with st.spinner(
        "Searching the research papers..."
    ):
        answer, sources = ask_question(
            question
        )
    st.markdown("### Answer")
    st.markdown(
        f'<div class="answer-box">'
        f'{answer}'
        f'</div>',

        unsafe_allow_html=True
    )
    if sources:
        st.markdown(
            "### 📄 Supporting Documents"
        )

        for i, (
            doc,
            score,
            paper_title
        ) in enumerate(
            sources,
            1
        ):

            page_number = doc.metadata.get(
                "page",
                "Unknown"
            )
            with st.expander(

                f"Source {i} — "
                f"{paper_title} — "
                f"Page {page_number}"

            ):
                st.markdown(
                    f"**Paper Title:** {paper_title}"
                )

                st.markdown(
                    f"**Page:** {page_number}"
                )

                st.markdown(
                    f"**Relevance Score:** "
                    f"{score:.4f}"
                )

                st.markdown(
                    "**Retrieved Chunk:**"
                )

                st.write(
                    doc.page_content
                )

# Step 24: Map Research Paper Names
    I map the research paper file identifiers to their actual paper titles

In [ ]:
paper_names = {
    "1706.03762": "Attention Is All You Need",
    "1810.04805": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
    "2005.11401": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
    "1908.10084": "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks",
    "2002.08909": "REALM: Retrieval-Augmented Language Model Pre-Training"
}

In [ ]:
test_source = "1706.03762v7 rp.pdf"
paper_title = next(
    (
        title
        for paper_id, title in paper_names.items()
        if paper_id in test_source
    ),
    "Unknown Research Paper"
)
print("Paper Title:", paper_title)

In [ ]:
def ask_question(question):
    paper_keywords = {
        "Sentence-BERT": [
            "sentence-bert",
            "sentence bert",
            "sbert",
            "siamese bert"
        ],
        "RAG": [
            "retrieval-augmented generation",
            "rag"
        ],
        "REALM": [
            "realm",
            "retrieval-augmented language model"
        ],
        "BERT": [
            "bert"
        ],
        "Transformer": [
            "transformer",
            "attention is all you need",
            "multi-head attention"
        ]
    }
    question_lower = question.lower()
    selected_paper = None
    for paper, keywords in paper_keywords.items():
        if any(keyword in question_lower for keyword in keywords):
            selected_paper = paper
            break
    if selected_paper is None:
        return (
            "The answer is not available in the provided research papers."
        ), []
    paper_ids = {
        "Sentence-BERT": "1908.10084",
        "RAG": "2005.11401",
        "REALM": "2002.08909",
        "BERT": "1810.04805",
        "Transformer": "1706.03762"
    }
    paper_id = paper_ids[selected_paper]
    selected_docs = [
        doc for doc in chunks
        if paper_id in doc.metadata.get("source", "")
    ]
    if not selected_docs:
        return (
            "The answer is not available in the provided research papers."
        ), []
    question_embedding = embedding_model.embed_query(question)
    scored_docs = []
    for doc in selected_docs:
        doc_embedding = embedding_model.embed_query(
            doc.page_content
        )
        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding)
            * np.linalg.norm(doc_embedding)
        )
        scored_docs.append((doc, similarity))
    scored_docs.sort(
        key=lambda x: x[1],
        reverse=True
    )
    relevant_docs = [
        (doc, score)
        for doc, score in scored_docs[:5]
        if score >= 0.30
    ]
    if not relevant_docs:
        return (
            "The answer is not available in the provided research papers."
        ), []
    context = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n"
            f"Page: {doc.metadata.get('page', 'Unknown')}\n"
            f"Content: {doc.page_content}"
            for doc, score in relevant_docs
        ]
    )
    messages = prompt.format_messages(
        context=context,
        question=question
    )

    response = llm.invoke(messages)
    answer = response.content.strip()
    if not answer:
        return (
            "The answer is not available in the provided research papers."
        ), []

    sources = []
    for doc, score in relevant_docs[:3]:
        source_file = doc.metadata.get(
            "source",
            "Unknown"
        )
        paper_title = next(
            (
                title
                for paper_id, title in paper_names.items()
                if paper_id in source_file
            ),
            "Unknown Research Paper"
        )
        sources.append(
            (doc, score, paper_title)
        )
    return answer, sources

In [ ]:
from pyngrok import ngrok
ngrok.kill()
print("Old ngrok tunnels stopped.")
